In [1]:
import pandas as pd

# Load the prepared forecasting dataset
df_model = pd.read_csv(
    "../data/processed/sovereign_spreads_model.csv"
)

# Check that the dataset loads correctly
print(df_model.shape)

(225, 19)


In [5]:
# Define our seven macroeconomic predictors
features = [
    "Debt_GDP",
    "Fiscal_Balance",
    "Real_GDP_Growth",
    "Inflation",
    "Unemployment",
    "Current_Account",
    "Interest_Payments"
]

lagged_features = [v + "_lag1" for v in features]

# Separate historical training data from testing data
train = df_model[df_model["Year"] <= 2019]
test = df_model[df_model["Year"] >= 2020]

# Define explanatory variables and target
X_train = train[lagged_features]
X_test = test[lagged_features]

print(X_train.shape, X_test.shape)

(171, 7) (54, 7)


In [8]:
from sklearn.linear_model import LinearRegression

# Create a Linear Regression model
linear_model = LinearRegression()

# Train the model using observations from 2001–2019
linear_model.fit(X_train, y_train)

print("Linear Regression trained successfully!")

Linear Regression trained successfully!


In [7]:
y_train = train["Spread_bp"]
y_test = test["Spread_bp"]

In [9]:
# Display the estimated coefficients
coefficients = pd.DataFrame({
    "Variable": lagged_features,
    "Coefficient": linear_model.coef_
})

print("Intercept:", linear_model.intercept_)
display(coefficients)

Intercept: -298.1636902705686


,Variable,Coefficient
0,Debt_GDP_lag1,3.097299
1,Fiscal_Balance_lag1,-16.776157
2,Real_GDP_Growth_lag1,-15.891535
3,Inflation_lag1,35.828800
4,Unemployment_lag1,11.323354
5,Current_Account_lag1,-2.758281
6,Interest_Payments_lag1,-17.866622


In [10]:
# Predict sovereign spreads for the test period (2020–2025)
y_pred_linear = linear_model.predict(X_test)

# Compare actual and predicted spreads
comparison = pd.DataFrame({
    "Country": test["Country"].values,
    "Year": test["Year"].values,
    "Actual_Spread": y_test.values,
    "Predicted_Spread": y_pred_linear
})

comparison.head(10)

,Country,Year,Actual_Spread,Predicted_Spread
0,AT,2020,24.900000,-38.786171
1,AT,2021,23.153333,282.054071
2,AT,2022,54.016667,120.614925
3,AT,2023,63.083333,263.153184
4,AT,2024,49.858333,306.073537
5,AT,2025,35.100833,159.260917
6,BE,2020,33.033333,71.286418
7,BE,2021,31.575000,313.100601
8,BE,2022,56.500000,179.449661
9,BE,2023,65.686667,425.567433


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Calculate prediction errors for Linear Regression
mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))

print(f"Linear Regression MAE: {mae_linear:.2f} bp")
print(f"Linear Regression RMSE: {rmse_linear:.2f} bp")

Linear Regression MAE: 194.33 bp
Linear Regression RMSE: 238.33 bp


In [12]:
# Use the previous year's spread as our naive prediction
y_pred_naive = test["Spread_bp_lag1"]

# Calculate forecasting errors
mae_naive = mean_absolute_error(y_test, y_pred_naive)
rmse_naive = np.sqrt(mean_squared_error(y_test, y_pred_naive))

print(f"Naive Benchmark MAE: {mae_naive:.2f} bp")
print(f"Naive Benchmark RMSE: {rmse_naive:.2f} bp")

Naive Benchmark MAE: 22.05 bp
Naive Benchmark RMSE: 32.79 bp


In [13]:
from sklearn.ensemble import GradientBoostingRegressor

# Create a Gradient Boosting model
boosting_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    random_state=42
)

# Train the model on historical observations
boosting_model.fit(X_train, y_train)

print("Gradient Boosting trained successfully!")

Gradient Boosting trained successfully!


In [14]:
# Predict sovereign spreads using Gradient Boosting
y_pred_boosting = boosting_model.predict(X_test)

# Calculate prediction errors
mae_boosting = mean_absolute_error(y_test, y_pred_boosting)
rmse_boosting = np.sqrt(mean_squared_error(y_test, y_pred_boosting))

print(f"Gradient Boosting MAE: {mae_boosting:.2f} bp")
print(f"Gradient Boosting RMSE: {rmse_boosting:.2f} bp")

Gradient Boosting MAE: 176.88 bp
Gradient Boosting RMSE: 353.80 bp


In [15]:
# Compare actual and predicted spreads
boosting_results = test[["Country", "Year", "Spread_bp"]].copy()

boosting_results["Prediction"] = y_pred_boosting

# Calculate absolute prediction errors
boosting_results["Absolute_Error"] = abs(
    boosting_results["Spread_bp"] - boosting_results["Prediction"]
)

# Display the five largest errors
boosting_results.nlargest(5, "Absolute_Error")

,Country,Year,Spread_bp,Prediction,Absolute_Error
120,GR,2021,122.650000,1587.561048,1464.911048
124,GR,2025,74.985833,954.683998,879.698164
123,GR,2024,99.916667,970.662514,870.745847
122,GR,2023,152.708333,1005.192428,852.484094
121,GR,2022,229.208333,995.838227,766.629894


In [16]:
# Select test observations excluding Greece
mask = test["Country"] != "GR"

# Evaluate existing predictions for the other countries
mae_no_gr = mean_absolute_error(
    y_test[mask], y_pred_boosting[mask.to_numpy()]
)

rmse_no_gr = np.sqrt(mean_squared_error(
    y_test[mask], y_pred_boosting[mask.to_numpy()]
))

print(f"MAE without Greece: {mae_no_gr:.2f} bp")
print(f"RMSE without Greece: {rmse_no_gr:.2f} bp")

MAE without Greece: 89.37 bp
RMSE without Greece: 181.68 bp


In [17]:
# Compare all models on the same sample, excluding Greece
predictions = {
    "Naive": y_pred_naive,
    "Linear Regression": y_pred_linear,
    "Gradient Boosting": y_pred_boosting
}

for name, pred in predictions.items():
    mae = mean_absolute_error(
        y_test[mask], pred[mask.to_numpy()]
    )
    rmse = np.sqrt(mean_squared_error(
        y_test[mask], pred[mask.to_numpy()]
    ))

    print(f"{name}: MAE = {mae:.2f}, RMSE = {rmse:.2f}")

Naive: MAE = 16.37, RMSE = 23.29
Linear Regression: MAE = 173.09, RMSE = 209.31
Gradient Boosting: MAE = 89.37, RMSE = 181.68


In [18]:
# Add the previous year's spread to our explanatory variables
features_with_spread = lagged_features + ["Spread_bp_lag1"]

X_train_extended = train[features_with_spread]
X_test_extended = test[features_with_spread]

# Train a new Linear Regression model
linear_extended = LinearRegression()
linear_extended.fit(X_train_extended, y_train)

# Evaluate on the test period
pred_extended = linear_extended.predict(X_test_extended)

mae_extended = mean_absolute_error(y_test, pred_extended)
rmse_extended = np.sqrt(mean_squared_error(y_test, pred_extended))

print(f"Extended OLS MAE: {mae_extended:.2f} bp")
print(f"Extended OLS RMSE: {rmse_extended:.2f} bp")

Extended OLS MAE: 83.78 bp
Extended OLS RMSE: 102.87 bp


In [19]:
# Train a regression using only the previous year's spread
ar_model = LinearRegression()

ar_model.fit(train[["Spread_bp_lag1"]], y_train)

# Predict the test period
pred_ar = ar_model.predict(test[["Spread_bp_lag1"]])

# Evaluate prediction errors
mae_ar = mean_absolute_error(y_test, pred_ar)
rmse_ar = np.sqrt(mean_squared_error(y_test, pred_ar))

print(f"AR(1) MAE: {mae_ar:.2f} bp")
print(f"AR(1) RMSE: {rmse_ar:.2f} bp")

AR(1) MAE: 27.92 bp
AR(1) RMSE: 33.66 bp


In [20]:
# Train Stochastic Gradient Boosting
sgb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.8,
    random_state=42
)

sgb_model.fit(X_train, y_train)

print("Stochastic Gradient Boosting trained successfully!")

Stochastic Gradient Boosting trained successfully!


In [21]:
# Predict sovereign spreads using SGB
y_pred_sgb = sgb_model.predict(X_test)

# Calculate prediction errors
mae_sgb = mean_absolute_error(y_test, y_pred_sgb)
rmse_sgb = np.sqrt(mean_squared_error(y_test, y_pred_sgb))

print(f"SGB MAE: {mae_sgb:.2f} bp")
print(f"SGB RMSE: {rmse_sgb:.2f} bp")

SGB MAE: 171.43 bp
SGB RMSE: 334.22 bp


In [22]:
from sklearn.preprocessing import StandardScaler

# Create a standardization tool
scaler = StandardScaler()

# Calculate the mean and standard deviation using training data only
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to the test data
X_test_scaled = scaler.transform(X_test)

print("Standardization completed")
print("Training data shape:", X_train_scaled.shape)

Standardization completed
Training data shape: (171, 7)


In [23]:
from sklearn.decomposition import PCA

# Create PCA using all seven components initially
pca = PCA(n_components=7)

# Fit PCA using the training data only
X_train_pca = pca.fit_transform(X_train_scaled)

# Show the variance explained by each component
for i, variance in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"PC{i}: {variance * 100:.2f}%")

PC1: 39.30%
PC2: 21.15%
PC3: 14.67%
PC4: 10.74%
PC5: 7.13%
PC6: 4.86%
PC7: 2.15%


In [24]:
# Retain the first four principal components
X_train_pca4 = X_train_pca[:, :4]

# Apply the trained PCA transformation to the test data
X_test_pca = pca.transform(X_test_scaled)
X_test_pca4 = X_test_pca[:, :4]

# Check the dimensions
print("Training:", X_train_pca4.shape)
print("Testing:", X_test_pca4.shape)

Training: (171, 4)
Testing: (54, 4)


In [25]:
# Train Linear Regression using four principal components
linear_pca = LinearRegression()

linear_pca.fit(X_train_pca4, y_train)

# Predict sovereign spreads for 2020–2025
y_pred_pca = linear_pca.predict(X_test_pca4)

# Calculate prediction errors
mae_pca = mean_absolute_error(y_test, y_pred_pca)
rmse_pca = np.sqrt(mean_squared_error(y_test, y_pred_pca))

print(f"OLS + PCA MAE: {mae_pca:.2f} bp")
print(f"OLS + PCA RMSE: {rmse_pca:.2f} bp")

OLS + PCA MAE: 128.64 bp
OLS + PCA RMSE: 187.47 bp
